# Maximal Spec-Decode 500U Lab

This notebook is the aggressive H100/G4 runway for the real gold mine: making one expensive verifier pass emit multiple useful tokens, while local Mac runtime wiring happens in parallel.

It is not a small-model vanity bench. It builds the artifacts needed to find the smallest still-useful general model and the strongest draft/spec stack around it.

Default lanes:

| Lane | Purpose | Why it exists |
|---|---|---|
| q1p5 flagship | Main past-200 candidate | Fast enough to matter, still plausibly useful. |
| q0p5 moonshot | Super-ceiling scout | Tests whether a tiny general student can stay coherent. |
| q3b reference | Quality anchor | Keeps the current Qwen-3B line honest. |

The notebook can also ingest the essential export zip/folder from the previous reconciliation run so already-trained q3b/q1p5 heads become seed candidates instead of lost inventory.

What Colab does here: corpora, frozen baselines, AWQ/Q2 artifacts, Eagle multi-block head sweeps, tau/frontier evaluation, leaderboards, export zips.

What stays local: Qwen dense runtime port, trace-dispatch acceptance proof, clean Mac benches, GGUF/profile generation, ship/no-ship decisions.


In [ ]:
# Cell 1 - Setup, budget, targets
from google.colab import drive
try:
    drive.mount('/content/drive')
except Exception as e:
    print(f'Drive mount warning: {e}')

import glob, gc, hashlib, json, math, os, shutil, signal, subprocess, sys, threading, time, zipfile
from pathlib import Path


def run(cmd, *, env=None, cwd=None, quiet=False):
    cmd = [str(x) for x in cmd]
    if cmd and cmd[0] == sys.executable and (len(cmd) == 1 or cmd[1] != '-u'):
        cmd.insert(1, '-u')
    merged = os.environ.copy()
    merged['PYTHONUNBUFFERED'] = '1'
    if env:
        merged.update(env)
    print('$ ' + ' '.join(cmd), flush=True)
    stdout = subprocess.DEVNULL if quiet else None
    subprocess.run(cmd, check=True, cwd=cwd, env=merged, stdout=stdout)


def run_with_heartbeat(cmd, label='job', interval_sec=30, cwd=None, env=None):
    cmd = [str(x) for x in cmd]
    if cmd and cmd[0] == sys.executable and (len(cmd) == 1 or cmd[1] != '-u'):
        cmd.insert(1, '-u')
    merged = os.environ.copy()
    merged['PYTHONUNBUFFERED'] = '1'
    if env:
        merged.update(env)
    print('$ ' + ' '.join(cmd), flush=True)
    proc = subprocess.Popen(cmd, cwd=cwd, env=merged)
    print(f'[{label}] pid={proc.pid} start={time.strftime("%H:%M:%S")}', flush=True)
    stop_evt = threading.Event()
    start = time.time()

    def heartbeat():
        while not stop_evt.wait(interval_sec):
            elapsed_m = (time.time() - start) / 60.0
            try:
                out = subprocess.check_output([
                    'nvidia-smi',
                    '--query-gpu=utilization.gpu,memory.used,memory.total',
                    '--format=csv,noheader,nounits',
                ], text=True, timeout=5).strip()
                util, used, total = [x.strip() for x in out.split(',')]
                gpu = f'gpu={util}% mem={used}/{total}MB'
            except Exception as e:
                gpu = f'gpu_query_failed={e}'
            state = 'RUNNING' if proc.poll() is None else f'EXITED rc={proc.returncode}'
            print(f'[{label}] {state} elapsed={elapsed_m:.1f}m {gpu}', flush=True)

    t = threading.Thread(target=heartbeat, daemon=True)
    t.start()
    try:
        rc = proc.wait()
    finally:
        stop_evt.set()
        t.join(timeout=2)
    elapsed_m = (time.time() - start) / 60.0
    print(f'[{label}] finished rc={rc} elapsed={elapsed_m:.1f}m', flush=True)
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)
    return elapsed_m


run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'accelerate>=1.0', 'datasets>=3.0', 'pyarrow>=17', 'tqdm>=4.66',
    'zstandard', 'bitsandbytes>=0.43', 'gguf', 'safetensors>=0.4',
    'hf_transfer>=0.1.9', 'transformers>=4.45',
])

os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER', '1')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('PYTHONUNBUFFERED', '1')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

try:
    from google.colab import userdata
    _hf_token = userdata.get('HF_TOKEN')
except Exception:
    _hf_token = None
if _hf_token:
    os.environ['HF_TOKEN'] = _hf_token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = _hf_token
    print('HF_TOKEN loaded from Colab secrets.')

import numpy as np
import torch
import transformers
from transformers import AutoConfig

assert torch.cuda.is_available(), 'No CUDA device. Pick an H100/G4/A100/L4 GPU runtime.'
GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
BIG_GPU = VRAM_GB >= 70
print(f'GPU: {GPU_NAME} VRAM={VRAM_GB:.1f}GB torch={torch.__version__} transformers={transformers.__version__}')
if not BIG_GPU:
    print('WARN: not a 70GB+ GPU. The notebook will still run, but reduce RUN_TARGET_ORDER or MAX_SPECS_PER_TARGET if needed.')

REPO = 'https://github.com/joshuahickscorp/dismantle.git'
BRANCH = 'codex/maximal-spec-colab'
REPO_DIR = Path('/content/dismantle')
DRIVE_ROOT = Path('/content/drive/MyDrive/dismantle')
LAB_ROOT = DRIVE_ROOT / 'maximal_spec_500u'
ARTIFACT_ROOT = LAB_ROOT / 'artifacts'
CORPUS_ROOT = LAB_ROOT / 'corpora'
EXPORT_ROOT = DRIVE_ROOT / 'maximal_spec_500u_export'
IMPORT_EXPORT_DIR = Path('/content/dismantle_export')
IMPORT_EXPORT_ZIP = Path('/content/dismantle_export.zip')
PROGRESS_PATH = LAB_ROOT / 'progress.json'

RUN_TARGET_ORDER = ['q1p5', 'q0p5', 'q3b_ref']
RUN_CORPUS = True
RUN_QUANT = True
RUN_TRAIN = True
RUN_EVAL = True
RUN_EXPORT = True
MAX_SPECS_PER_TARGET = {'q1p5': 10, 'q0p5': 12, 'q3b_ref': 5}
TAU_DEPTH = 8
FRONTIER_DEPTHS = '2,4,6,8,12,16,24'
FRONTIER_WIDTHS = '2,3,4,6,8'

QWEN_LOCKED_ENV = {
    'DISMANTLE_QWEN_TCB': '1',
    'DISMANTLE_QWEN_VOCAB_PRUNE': '32000',
    'DISMANTLE_QWEN_Q4K_LMHEAD': '1',
    'DISMANTLE_QWEN_FFN_DOWN_Q4K': '1',
    'DISMANTLE_QWEN_Q4K_PREDEC': '1',
}

TARGETS = {
    'q1p5': {
        'name': 'q1p5',
        'model_id': 'Qwen/Qwen2.5-1.5B-Instruct',
        'track': 'flagship_fast_general',
        'corpus_target_seqs': 24000 if BIG_GPU else 8000,
        'train_max_rows': 24000 if BIG_GPU else 8000,
        'train_max_row_tokens': 384 if BIG_GPU else 192,
        'train_epochs': 18 if BIG_GPU else 10,
        'calibration_batch': 16 if BIG_GPU else 6,
        'lm_head_chunk_tokens': 512 if BIG_GPU else 256,
        'eval_max_windows': 24000 if BIG_GPU else 6000,
        'frontier_max_depth': 24 if BIG_GPU else 12,
        'base_tps_placeholder': 95.0,
        'spec_efficiency_placeholder': 0.80,
        'gguf_name': 'qwen2.5-1.5b-instruct-q4_k_m.gguf',
        'profile_name': 'qwen15b-instruct-q4k.m3pro18.json',
    },
    'q0p5': {
        'name': 'q0p5',
        'model_id': 'Qwen/Qwen2.5-0.5B-Instruct',
        'track': 'moonshot_tiny_general',
        'corpus_target_seqs': 36000 if BIG_GPU else 12000,
        'train_max_rows': 36000 if BIG_GPU else 12000,
        'train_max_row_tokens': 384 if BIG_GPU else 192,
        'train_epochs': 20 if BIG_GPU else 12,
        'calibration_batch': 24 if BIG_GPU else 8,
        'lm_head_chunk_tokens': 768 if BIG_GPU else 256,
        'eval_max_windows': 32000 if BIG_GPU else 8000,
        'frontier_max_depth': 24 if BIG_GPU else 12,
        'base_tps_placeholder': 180.0,
        'spec_efficiency_placeholder': 0.78,
        'gguf_name': 'qwen2.5-0.5b-instruct-q4_k_m.gguf',
        'profile_name': 'qwen05b-instruct-q4k.m3pro18.json',
    },
    'q3b_ref': {
        'name': 'q3b_ref',
        'model_id': 'Qwen/Qwen2.5-3B-Instruct',
        'track': 'quality_anchor_reference',
        'corpus_target_seqs': 8000 if BIG_GPU else 3000,
        'train_max_rows': 8000 if BIG_GPU else 3000,
        'train_max_row_tokens': 192,
        'train_epochs': 10 if BIG_GPU else 6,
        'calibration_batch': 8 if BIG_GPU else 3,
        'lm_head_chunk_tokens': 384 if BIG_GPU else 128,
        'eval_max_windows': 10000 if BIG_GPU else 4000,
        'frontier_max_depth': 16 if BIG_GPU else 8,
        'base_tps_placeholder': 30.0,
        'spec_efficiency_placeholder': 0.72,
        'gguf_name': 'qwen2.5-3b-instruct-q4_k_m.gguf',
        'profile_name': 'qwen3b-instruct-q4k.m3pro18.json',
        'corpus_aliases': [DRIVE_ROOT / 'qwen3b_corpus'],
    },
}

TRAIN_GRID = [
    {'tag': 'b1_fast',    'num_blocks': 1, 'head_heads': 16, 'head_ff_mult': 4.0, 'seq_len': 16, 'lr': 3e-4, 'rd': 0.00, 'cw': 0.12, 'seed': 0, 'targets': ['q1p5','q0p5','q3b_ref']},
    {'tag': 'b1_wide',    'num_blocks': 1, 'head_heads': 16, 'head_ff_mult': 6.0, 'seq_len': 16, 'lr': 5e-4, 'rd': 0.02, 'cw': 0.20, 'seed': 0, 'targets': ['q1p5','q0p5','q3b_ref']},
    {'tag': 'b1_hot',     'num_blocks': 1, 'head_heads': 16, 'head_ff_mult': 4.0, 'seq_len': 16, 'lr': 1e-3, 'rd': 0.01, 'cw': 0.16, 'seed': 1, 'targets': ['q1p5','q0p5','q3b_ref']},
    {'tag': 'b2_wide',    'num_blocks': 2, 'head_heads': 16, 'head_ff_mult': 6.0, 'seq_len': 16, 'lr': 5e-4, 'rd': 0.02, 'cw': 0.20, 'seed': 0, 'targets': ['q1p5','q0p5','q3b_ref']},
    {'tag': 'b2_hot',     'num_blocks': 2, 'head_heads': 16, 'head_ff_mult': 4.0, 'seq_len': 16, 'lr': 1e-3, 'rd': 0.02, 'cw': 0.22, 'seed': 1, 'targets': ['q1p5','q0p5']},
    {'tag': 'b2_longctx', 'num_blocks': 2, 'head_heads': 16, 'head_ff_mult': 6.0, 'seq_len': 24, 'lr': 3e-4, 'rd': 0.03, 'cw': 0.24, 'seed': 2, 'targets': ['q1p5','q0p5']},
    {'tag': 'b3_compact', 'num_blocks': 3, 'head_heads': 16, 'head_ff_mult': 4.0, 'seq_len': 16, 'lr': 3e-4, 'rd': 0.03, 'cw': 0.24, 'seed': 0, 'targets': ['q1p5','q0p5']},
    {'tag': 'b3_wide',    'num_blocks': 3, 'head_heads': 16, 'head_ff_mult': 6.0, 'seq_len': 16, 'lr': 5e-4, 'rd': 0.03, 'cw': 0.26, 'seed': 1, 'targets': ['q1p5','q0p5']},
    {'tag': 'b4_tiny',    'num_blocks': 4, 'head_heads': 16, 'head_ff_mult': 4.0, 'seq_len': 16, 'lr': 3e-4, 'rd': 0.04, 'cw': 0.28, 'seed': 2, 'targets': ['q0p5']},
    {'tag': 'b4_wide',    'num_blocks': 4, 'head_heads': 16, 'head_ff_mult': 6.0, 'seq_len': 16, 'lr': 5e-4, 'rd': 0.04, 'cw': 0.30, 'seed': 3, 'targets': ['q0p5']},
    {'tag': 'b2_seed2',   'num_blocks': 2, 'head_heads': 16, 'head_ff_mult': 6.0, 'seq_len': 16, 'lr': 5e-4, 'rd': 0.02, 'cw': 0.20, 'seed': 2, 'targets': ['q1p5','q0p5']},
    {'tag': 'b3_seed3',   'num_blocks': 3, 'head_heads': 16, 'head_ff_mult': 6.0, 'seq_len': 16, 'lr': 4e-4, 'rd': 0.035, 'cw': 0.26, 'seed': 3, 'targets': ['q1p5','q0p5']},
]

for p in [DRIVE_ROOT, LAB_ROOT, ARTIFACT_ROOT, CORPUS_ROOT, EXPORT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print(json.dumps({
    'gpu': GPU_NAME,
    'vram_gb': VRAM_GB,
    'big_gpu': BIG_GPU,
    'run_target_order': RUN_TARGET_ORDER,
    'max_specs_per_target': MAX_SPECS_PER_TARGET,
    'lab_root': str(LAB_ROOT),
    'import_export_dir': str(IMPORT_EXPORT_DIR),
}, indent=2))


In [ ]:
# Cell 2 - Repo, progress, export ingest helpers

def sha256_file(path, bs=8 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            b = f.read(bs)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def load_json(path, default=None):
    path = Path(path)
    if path.exists():
        try:
            return json.loads(path.read_text())
        except Exception as e:
            print(f'WARN: could not parse {path}: {e}')
    return {} if default is None else default


def write_json_atomic(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(payload, indent=2, sort_keys=True, default=str) + '\n')
    os.replace(tmp, path)


def progress():
    return load_json(PROGRESS_PATH, {})


def save_progress(p):
    write_json_atomic(PROGRESS_PATH, p)


def record_stage(stage, **payload):
    p = progress()
    payload.setdefault('completed_at_unix', int(time.time()))
    payload.setdefault('repo_sha', globals().get('HEAD_SHA', 'unknown'))
    p[stage] = payload
    save_progress(p)
    print(f'[progress] {stage}: recorded')


def kill_stale(pattern):
    try:
        out = subprocess.check_output(['pgrep', '-f', pattern], text=True).strip()
    except subprocess.CalledProcessError:
        return
    pids = [int(x) for x in out.splitlines() if x.strip().isdigit()]
    if not pids:
        return
    print(f'[kill-stale] {pattern}: {pids}')
    for pid in pids:
        try:
            os.kill(pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
    time.sleep(3)
    for pid in pids:
        try:
            os.kill(pid, signal.SIGKILL)
        except ProcessLookupError:
            pass


def env_line(env):
    return ' '.join(f'{k}={v}' for k, v in env.items())


if not REPO_DIR.exists():
    run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO, str(REPO_DIR)])
else:
    run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH, '--depth', '1'])
    run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH])
    run(['git', '-C', str(REPO_DIR), 'reset', '--hard', f'origin/{BRANCH}'])
os.chdir(REPO_DIR)
HEAD_SHA = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
print(f'repo @ {HEAD_SHA}')

REQUIRED = [
    'colab/mega_calibrate.py',
    'colab/eagle5_train_pytorch.py',
    'colab/eagle5_tau_eval_pytorch.py',
    'colab/eagle5_frontier_policy.py',
    'colab/build_qwen3b_frozen_hf.py',
    'colab/q2k_importance_calibrate.py',
    'colab/awq_per_channel_calibrate.py',
    'tools/training/awq_calibrate.py',
    'colab/export_reconciliation_essentials.py',
]
missing = [p for p in REQUIRED if not Path(p).exists()]
if missing:
    raise RuntimeError(f'Missing required repo files on branch {BRANCH}: {missing}')

IMPORTED_HEADS = {'q1p5': [], 'q0p5': [], 'q3b_ref': []}
def ingest_export_inventory():
    if IMPORT_EXPORT_ZIP.exists() and not IMPORT_EXPORT_DIR.exists():
        print(f'unzipping prior export: {IMPORT_EXPORT_ZIP}')
        with zipfile.ZipFile(IMPORT_EXPORT_ZIP) as zf:
            zf.extractall(IMPORT_EXPORT_DIR)
    manifest = IMPORT_EXPORT_DIR / 'manifest.json'
    if not manifest.exists():
        print('No prior export manifest found. Continuing fresh.')
        return
    data = load_json(manifest, {})
    imported_root = LAB_ROOT / 'imported_export'
    imported_root.mkdir(parents=True, exist_ok=True)
    shutil.copy2(manifest, imported_root / 'manifest.json')
    for name, info in data.get('files', {}).items():
        exported = Path(info.get('exported', ''))
        if not exported.exists():
            # Export manifests often contain absolute paths from the old
            # Colab runtime. After uploading/unzipping, rediscover by basename.
            matches = list(IMPORT_EXPORT_DIR.rglob(exported.name))
            exported = matches[0] if matches else exported
        if not exported.exists() or not str(exported).endswith('.safetensors'):
            continue
        dst = imported_root / 'heads' / exported.name
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists() or dst.stat().st_size != exported.stat().st_size:
            shutil.copy2(exported, dst)
        if 'q1p5' in name or '1p5' in exported.name:
            IMPORTED_HEADS['q1p5'].append(dst)
        elif 'q3b' in name or '3b' in exported.name:
            IMPORTED_HEADS['q3b_ref'].append(dst)
    print('Imported heads:', {k: [str(x) for x in v] for k, v in IMPORTED_HEADS.items()})

ingest_export_inventory()
print('Existing progress stages:', sorted(progress().keys()))


In [ ]:
# Cell 2R - Cold resume bootstrap from Drive artifacts
# Use this after a Colab reconnect/reset. It reconstructs notebook state from
# Drive and the checked-out GitHub branch without rerunning expensive corpus prep.
# Edit RESUME_TARGETS before running when this runtime is a worker for another lane.
RESUME_TARGETS = globals().get('RESUME_TARGETS', ['q1p5'])

from google.colab import drive
try:
    drive.mount('/content/drive')
except Exception as e:
    print(f'Drive mount warning: {e}')

import gc, hashlib, json, math, os, shutil, signal, subprocess, sys, threading, time, zipfile
from pathlib import Path

REPO = 'https://github.com/joshuahickscorp/dismantle.git'
BRANCH = 'codex/maximal-spec-colab'
REPO_DIR = Path('/content/dismantle')


def _bootstrap_run(cmd, *, cwd=None):
    cmd = [str(x) for x in cmd]
    print('$ ' + ' '.join(cmd), flush=True)
    subprocess.run(cmd, check=True, cwd=cwd)


if not REPO_DIR.exists():
    _bootstrap_run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO, str(REPO_DIR)])
else:
    _bootstrap_run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH, '--depth', '1'])
    _bootstrap_run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH])
    _bootstrap_run(['git', '-C', str(REPO_DIR), 'reset', '--hard', f'origin/{BRANCH}'])
os.chdir(REPO_DIR)

_nb_path = REPO_DIR / 'colab/maximal_spec_decode_500u.ipynb'
_nb = json.loads(_nb_path.read_text())

def _cell_source(prefix):
    for cell in _nb['cells']:
        src = ''.join(cell.get('source', []))
        if src.startswith(prefix):
            return src
    raise RuntimeError(f'Could not find notebook cell starting with {prefix!r}')


# Re-exec the canonical setup/helper cells from the checked-out notebook so
# definitions stay in one place. Cell 1 installs deps and defines targets/grid;
# Cell 2 defines repo/progress helpers and imported-head inventory.
exec(_cell_source('# Cell 1 - Setup'), globals())
exec(_cell_source('# Cell 2 - Repo'), globals())

# Define Cell 3 helper functions without running its expensive target loop.
_cell3 = _cell_source('# Cell 3 - Build/resume')
exec(_cell3.split('TARGET_STATE = {}', 1)[0], globals())

RUN_TARGET_ORDER = list(RESUME_TARGETS)
PROGRESS_PATH = LAB_ROOT / ('progress_' + '_'.join(RUN_TARGET_ORDER) + '_resume.json')
RUN_CORPUS = False
RUN_QUANT = True
RUN_TRAIN = True
RUN_EVAL = True
RUN_EXPORT = False
IMPORTED_HEADS = globals().get('IMPORTED_HEADS', {'q1p5': [], 'q0p5': [], 'q3b_ref': []})


def _expected_quant_outputs(t, artifact_dir, corpus_dir):
    stats = corpus_dir / 'per_site_activation_stats.npz'
    return {
        'stats': str(stats),
        'awq_fixed': str(artifact_dir / f'{t["name"]}_awq_fixed_alpha05.json'),
        'awq_adaptive': str(artifact_dir / f'{t["name"]}_awq_adaptive_alpha.json'),
        'awq_clipped': str(artifact_dir / f'{t["name"]}_awq_outlier_clipped.json'),
        'q2_importance': str(artifact_dir / f'{t["name"]}_q2k_importance.npz'),
    }


def rehydrate_target_state(name):
    t = TARGETS[name]
    artifact_dir, corpus_dir, local_corpus, frozen, local_frozen = target_paths(t)
    info = target_config(t)
    state = corpus_state(t, corpus_dir)
    print(f'[resume:{name}] corpus state: {state} path={corpus_dir}')
    if not state['complete']:
        raise RuntimeError(
            f'{name} corpus incomplete in Drive: {state}. Run Cell 3 for this target first.'
        )
    if not frozen.exists():
        raise RuntimeError(f'{name} frozen baseline missing: {frozen}')
    if not local_frozen.exists() or local_frozen.stat().st_size != frozen.stat().st_size:
        print(f'[resume:{name}] stage frozen -> {local_frozen}')
        shutil.copyfile(frozen, local_frozen)
    quant = _expected_quant_outputs(t, artifact_dir, corpus_dir)
    missing_quant = [k for k, v in quant.items() if k != 'stats' and not Path(v).exists()]
    if missing_quant:
        print(f'[resume:{name}] WARN missing quant artifacts: {missing_quant}')
    return {
        'target': t,
        'info': info,
        'artifact_dir': str(artifact_dir),
        'corpus_dir': str(corpus_dir),
        'local_frozen': str(local_frozen),
        'frozen': str(frozen),
        'quant': quant,
    }


TARGET_STATE = {name: rehydrate_target_state(name) for name in RUN_TARGET_ORDER}
print('[resume] RUN_TARGET_ORDER =', RUN_TARGET_ORDER)
print('[resume] PROGRESS_PATH =', PROGRESS_PATH)
for name, state in TARGET_STATE.items():
    ckpt_root = Path(state['artifact_dir']) / 'checkpoints'
    completed = sorted(p.parent.name for p in ckpt_root.glob('*/head_final.safetensors'))
    partial = sorted(p.parent.name for p in ckpt_root.glob('*/latest.npz') if not (p.parent / 'head_final.safetensors').exists())
    print(f'[resume:{name}] completed heads={len(completed)} partial heads={len(partial)}')
    for tag in completed[:20]:
        print('  final  ', tag)
    for tag in partial[:20]:
        print('  partial', tag)


In [ ]:
# Cell 3 - Build/resume corpora, frozen baselines, and quant artifacts

def target_paths(t):
    name = t['name']
    artifact_dir = ARTIFACT_ROOT / name
    artifact_dir.mkdir(parents=True, exist_ok=True)
    corpus_dir = CORPUS_ROOT / f'{name}_corpus'
    aliases = [Path(x) for x in t.get('corpus_aliases', [])]
    for alias in aliases:
        if alias.exists() and len(list(alias.glob('shard_*.parquet'))) > len(list(corpus_dir.glob('shard_*.parquet'))):
            corpus_dir = alias
            break
    local_corpus = Path('/content') / f'{name}_corpus'
    local_frozen = Path('/content') / f'{name}_frozen.npz'
    frozen = artifact_dir / f'{name}_frozen.npz'
    return artifact_dir, corpus_dir, local_corpus, frozen, local_frozen


def target_config(t):
    cfg = AutoConfig.from_pretrained(t['model_id'], trust_remote_code=False)
    n_layers = int(getattr(cfg, 'num_hidden_layers', 0) or getattr(cfg, 'n_layer', 0))
    hidden = int(getattr(cfg, 'hidden_size', 0) or getattr(cfg, 'n_embd', 0))
    vocab = int(getattr(cfg, 'vocab_size', 0))
    capture_layer = max(0, n_layers - 4)
    return {'n_layers': n_layers, 'hidden': hidden, 'vocab': vocab, 'capture_layer': capture_layer}


def stats_sequences(path):
    if not Path(path).exists():
        return 0
    try:
        with np.load(path) as z:
            return int(np.asarray(z['sequences_written']).item()) if 'sequences_written' in z.files else 0
    except Exception as e:
        print(f'WARN: could not read {path}: {e}')
        return 0


def corpus_state(t, corpus_dir):
    stats = corpus_dir / 'per_site_activation_stats.npz'
    shards = len(list(corpus_dir.glob('shard_*.parquet')))
    seqs = stats_sequences(stats)
    need = math.ceil(t['corpus_target_seqs'] / 16)
    return {'shards': shards, 'seqs': seqs, 'required_shards': need, 'complete': shards >= need and seqs >= t['corpus_target_seqs']}


def build_corpus(t, info, corpus_dir, local_corpus):
    state = corpus_state(t, corpus_dir)
    print(f'[{t["name"]}] corpus state: {state} path={corpus_dir}')
    if state['complete']:
        return
    if not RUN_CORPUS:
        raise RuntimeError(f'{t["name"]} corpus incomplete and RUN_CORPUS=False')
    corpus_dir.mkdir(parents=True, exist_ok=True)
    local_corpus.mkdir(parents=True, exist_ok=True)
    run_with_heartbeat([
        sys.executable, 'colab/mega_calibrate.py',
        '--model', t['model_id'],
        '--max-sequences', str(t['corpus_target_seqs']),
        '--max-tokens', '2048',
        '--batch-size', str(t['calibration_batch']),
        '--capture-layer', str(info['capture_layer']),
        '--topk-logits', '64',
        '--lm-head-chunk-tokens', str(t['lm_head_chunk_tokens']),
        '--shard-size', '16',
        '--sync-dir', str(corpus_dir),
        '--sync-every', '4',
        '--delete-local-after-sync',
        '--out', str(local_corpus),
    ], label=f'corpus-{t["name"]}', interval_sec=60)
    final_state = corpus_state(t, corpus_dir)
    if not final_state['complete']:
        raise RuntimeError(f'corpus still incomplete for {t["name"]}: {final_state}')


def build_frozen(t, frozen, local_frozen):
    if not frozen.exists():
        run_with_heartbeat([
            sys.executable, 'colab/build_qwen3b_frozen_hf.py',
            '--model', t['model_id'],
            '--out', str(frozen),
        ], label=f'frozen-{t["name"]}', interval_sec=60)
    if not frozen.exists():
        raise RuntimeError(f'missing frozen baseline: {frozen}')
    if not local_frozen.exists() or local_frozen.stat().st_size != frozen.stat().st_size:
        print(f'[{t["name"]}] stage frozen -> {local_frozen}')
        shutil.copyfile(frozen, local_frozen)
    return local_frozen


def build_quant(t, corpus_dir, artifact_dir):
    stats = corpus_dir / 'per_site_activation_stats.npz'
    if not stats.exists():
        raise RuntimeError(f'missing stats for quant artifacts: {stats}')
    outputs = {'stats': str(stats)}
    if not RUN_QUANT:
        return outputs
    fixed = artifact_dir / f'{t["name"]}_awq_fixed_alpha05.json'
    adaptive = artifact_dir / f'{t["name"]}_awq_adaptive_alpha.json'
    clipped = artifact_dir / f'{t["name"]}_awq_outlier_clipped.json'
    q2 = artifact_dir / f'{t["name"]}_q2k_importance.npz'
    if not fixed.exists():
        run([sys.executable, 'tools/training/awq_calibrate.py', '--stats', str(stats), '--out', str(fixed), '--alpha', '0.5', '--model', t['model_id']])
    if not adaptive.exists():
        run([sys.executable, 'colab/awq_per_channel_calibrate.py', '--stats', str(stats), '--out', str(adaptive), '--mode', 'adaptive-alpha', '--model', t['model_id']])
    if not clipped.exists():
        run([sys.executable, 'colab/awq_per_channel_calibrate.py', '--stats', str(stats), '--out', str(clipped), '--mode', 'outlier-clipped', '--clip-percentile', '99.5', '--model', t['model_id']])
    if not q2.exists():
        run([sys.executable, 'colab/q2k_importance_calibrate.py', '--stats', str(stats), '--out', str(q2), '--model', t['model_id'], '--mean-weight', '0.70', '--max-weight', '0.30'])
    outputs.update({'awq_fixed': str(fixed), 'awq_adaptive': str(adaptive), 'awq_clipped': str(clipped), 'q2_importance': str(q2)})
    return outputs


TARGET_STATE = {}
for name in RUN_TARGET_ORDER:
    t = TARGETS[name]
    print(f'\n=== Preparing {name}: {t["model_id"]}')
    artifact_dir, corpus_dir, local_corpus, frozen, local_frozen = target_paths(t)
    info = target_config(t)
    build_corpus(t, info, corpus_dir, local_corpus)
    local_frozen = build_frozen(t, frozen, local_frozen)
    quant = build_quant(t, corpus_dir, artifact_dir)
    TARGET_STATE[name] = {
        'target': t,
        'info': info,
        'artifact_dir': str(artifact_dir),
        'corpus_dir': str(corpus_dir),
        'local_frozen': str(local_frozen),
        'frozen': str(frozen),
        'quant': quant,
    }
    record_stage(f'prepare_{name}', state=TARGET_STATE[name])

print(json.dumps(TARGET_STATE, indent=2, sort_keys=True, default=str))


In [ ]:
# Cell 4 - Train maximal Eagle/MTP-style head grid

if 'TARGET_STATE' not in globals():
    raise RuntimeError('Runtime state missing. Run Cell 2R - Cold resume bootstrap, then rerun Cell 4.')


def lr_tag(lr):
    return f'{lr:.0e}'.replace('+', '').replace('-0', '-')


def spec_name(spec):
    rd = int(round(spec['rd'] * 1000))
    cw = int(round(spec['cw'] * 100))
    ff = int(round(spec['head_ff_mult'] * 10))
    return f"{spec['tag']}_b{spec['num_blocks']}_h{spec['head_heads']}_ff{ff:02d}_s{spec['seq_len']}_lr{lr_tag(spec['lr'])}_rd{rd:03d}_cw{cw:02d}_seed{spec['seed']}"


def train_batch_for(target_name, spec):
    if target_name == 'q0p5':
        return 96 if BIG_GPU else 24
    if target_name == 'q1p5':
        return 56 if BIG_GPU else 16
    return 24 if BIG_GPU else 8


def candidate_specs_for(target_name):
    specs = [s for s in TRAIN_GRID if target_name in s['targets']]
    return specs[:MAX_SPECS_PER_TARGET.get(target_name, len(specs))]


IMPORTED_HEADS = globals().get('IMPORTED_HEADS', {'q1p5': [], 'q0p5': [], 'q3b_ref': []})
TRAINED_HEADS = {k: list(v) for k, v in IMPORTED_HEADS.items()}
if RUN_TRAIN:
    for name in RUN_TARGET_ORDER:
        state = TARGET_STATE[name]
        t = state['target']
        artifact_dir = Path(state['artifact_dir'])
        ckpt_root = artifact_dir / 'checkpoints'
        ckpt_root.mkdir(parents=True, exist_ok=True)
        for spec in candidate_specs_for(name):
            tag = spec_name(spec)
            ckpt_dir = ckpt_root / f'{name}_{tag}'
            head = ckpt_dir / 'head_final.safetensors'
            stage = f'train_{name}_{tag}'
            if head.exists():
                print(f'[{stage}] skip existing {head}')
                TRAINED_HEADS.setdefault(name, []).append(head)
                continue
            kill_stale('eagle5_train_pytorch.py')
            gc.collect()
            torch.cuda.empty_cache()
            ckpt_dir.mkdir(parents=True, exist_ok=True)
            batch = train_batch_for(name, spec)
            print(f'\n=== [{name}] train {tag} batch={batch}')
            elapsed = run_with_heartbeat([
                sys.executable, 'colab/eagle5_train_pytorch.py',
                '--corpus-dir', state['corpus_dir'],
                '--frozen', state['local_frozen'],
                '--ckpt-dir', str(ckpt_dir),
                '--epochs', str(t['train_epochs']),
                '--batch-size', str(batch),
                '--seq-len', str(spec['seq_len']),
                '--lr', str(spec['lr']),
                '--num-blocks', str(spec['num_blocks']),
                '--head-heads', str(spec['head_heads']),
                '--head-ff-mult', str(spec['head_ff_mult']),
                '--capture-layer', str(state['info']['capture_layer']),
                '--max-rows', str(t['train_max_rows']),
                '--max-row-tokens', str(t['train_max_row_tokens']),
                '--sparsity-head', 'off',
                '--seed', str(spec['seed']),
                '--calib-loss-weight', str(spec['cw']),
                '--residual-delta-loss-weight', str(spec['rd']),
                '--save-safetensors',
            ], label=stage, interval_sec=60)
            if not head.exists():
                raise RuntimeError(f'training failed to write {head}')
            sha = sha256_file(head)
            record_stage(stage, path=str(head), size=head.stat().st_size, sha256=sha, minutes=round(elapsed, 1), spec=spec)
            TRAINED_HEADS.setdefault(name, []).append(head)
else:
    print('RUN_TRAIN=False; only imported heads will be evaluated.')

for name, heads in TRAINED_HEADS.items():
    print(f'{name}: {len(heads)} head(s)')
    for h in heads:
        print('  ', h)


In [ ]:
# Cell 5 - Tau/frontier evaluation and leaderboard

def read_metadata_from_head(head_path):
    meta = {}
    try:
        from safetensors import safe_open
        with safe_open(str(head_path), framework='pt', device='cpu') as f:
            meta = f.metadata() or {}
    except Exception as e:
        print(f'WARN: metadata read failed for {head_path}: {e}')
    return meta


def eval_head(name, head):
    state = TARGET_STATE[name]
    t = state['target']
    head = Path(head)
    tag = head.parent.name if head.parent.name != 'heads' else head.stem
    out_dir = Path(state['artifact_dir']) / 'eval' / tag
    out_dir.mkdir(parents=True, exist_ok=True)
    tau_path = out_dir / 'tau.json'
    frontier_path = out_dir / 'frontier.json'
    meta = read_metadata_from_head(head)
    nb = meta.get('num_blocks', '1')
    hh = meta.get('n_heads', '16')
    ff = meta.get('ff_mult', '4.0')

    if not tau_path.exists():
        run_with_heartbeat([
            sys.executable, 'colab/eagle5_tau_eval_pytorch.py',
            '--ckpt', str(head),
            '--frozen', state['local_frozen'],
            '--corpus', state['corpus_dir'],
            '--out', str(tau_path),
            '--depth', str(min(TAU_DEPTH, t['frontier_max_depth'])),
            '--max-windows', str(t['eval_max_windows']),
            '--max-row-tokens', str(t['train_max_row_tokens']),
            '--num-blocks', str(nb),
            '--head-heads', str(hh),
            '--head-ff-mult', str(ff),
            '--base-tps', str(t['base_tps_placeholder']),
            '--w4a8-multiplier', '1.0',
            '--spec-efficiency', str(t['spec_efficiency_placeholder']),
        ], label=f'tau-{name}-{tag}', interval_sec=60)
    if not frontier_path.exists():
        run_with_heartbeat([
            sys.executable, 'colab/eagle5_frontier_policy.py',
            '--ckpt', str(head),
            '--frozen', state['local_frozen'],
            '--corpus', state['corpus_dir'],
            '--out', str(frontier_path),
            '--max-depth', str(t['frontier_max_depth']),
            '--depths', FRONTIER_DEPTHS,
            '--lattice-widths', FRONTIER_WIDTHS,
            '--max-windows', str(t['eval_max_windows']),
            '--max-row-tokens', str(t['train_max_row_tokens']),
            '--eval-batch-size', '192',
            '--num-blocks', str(nb),
            '--head-heads', str(hh),
            '--head-ff-mult', str(ff),
            '--base-tps', str(t['base_tps_placeholder']),
            '--w4a8-multiplier', '1.0',
            '--spec-efficiency', str(t['spec_efficiency_placeholder']),
        ], label=f'frontier-{name}-{tag}', interval_sec=60)
    tau = load_json(tau_path, {})
    frontier = load_json(frontier_path, {})
    best = frontier.get('policies', {}).get('best_deployable', {})
    overall = frontier.get('policies', {}).get('best_overall', {})
    return {
        'target': name,
        'tag': tag,
        'head': str(head),
        'tau_path': str(tau_path),
        'frontier_path': str(frontier_path),
        'tau': tau.get('tau'),
        'depth1_accept_rate': tau.get('depth1_accept_rate'),
        'best_deployable': best,
        'best_overall': overall,
        'offline_projected_tps': best.get('projected_dec_tps', 0.0),
        'accepted_draft_tokens_per_verify': best.get('accepted_draft_tokens_per_verify', 0.0),
        'policy_kind': best.get('kind'),
        'metadata': read_metadata_from_head(head),
    }


LEADERBOARD = []
if RUN_EVAL:
    for name in RUN_TARGET_ORDER:
        for head in TRAINED_HEADS.get(name, []):
            LEADERBOARD.append(eval_head(name, head))
else:
    print('RUN_EVAL=False; skipping tau/frontier.')

LEADERBOARD = sorted(LEADERBOARD, key=lambda r: (float(r.get('offline_projected_tps') or 0.0), float(r.get('tau') or 0.0)), reverse=True)
leaderboard_path = LAB_ROOT / 'leaderboard.json'
write_json_atomic(leaderboard_path, {'schema': 'dismantle-maximal-spec-leaderboard-v1', 'rows': LEADERBOARD})
record_stage('leaderboard', path=str(leaderboard_path), size=leaderboard_path.stat().st_size)

print('\n=== Leaderboard ===')
for r in LEADERBOARD[:30]:
    print(f"{r['target']:7s} {r['tag'][:48]:48s} tau={r.get('tau')} acc1={r.get('depth1_accept_rate')} accepted={r.get('accepted_draft_tokens_per_verify')} offline_tps={r.get('offline_projected_tps')} policy={r.get('policy_kind')}")


In [ ]:
# Cell 6 - Maximal summary, local handoff, and export zip

def best_for(target_name):
    rows = [r for r in LEADERBOARD if r['target'] == target_name]
    return rows[0] if rows else None


def zip_dir(src_dir, zip_path):
    zip_path = Path(zip_path)
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for p in sorted(Path(src_dir).rglob('*')):
            if p.is_file() and p != zip_path:
                zf.write(p, p.relative_to(src_dir))


summary = {
    'schema': 'dismantle-maximal-spec-500u-summary-v1',
    'repo_sha': HEAD_SHA,
    'gpu': GPU_NAME,
    'vram_gb': VRAM_GB,
    'big_gpu': BIG_GPU,
    'targets': {},
    'leaderboard_path': str(LAB_ROOT / 'leaderboard.json'),
    'locked_qwen_env': QWEN_LOCKED_ENV,
}

lines = [
    '# Maximal Spec-Decode 500U Summary',
    '',
    f'GPU: {GPU_NAME} ({VRAM_GB:.1f} GB)',
    f'Repo: {HEAD_SHA}',
    '',
    '## What This Run Optimized',
    '',
    '- Draft/spec heads that can make one verifier pass emit multiple tokens.',
    '- Smaller but still general Qwen students instead of fake speed-only toy paths.',
    '- Quant/calibration inventory for future local Metal experiments.',
    '- Offline policy ranking. Local Mac trace-dispatch remains the real proof.',
    '',
    '## Winners',
    '',
    '| Target | Track | Winner | Policy | Tau | Accepted/Verify | Offline TPS |',
    '|---|---|---|---|---:|---:|---:|',
]

for name in RUN_TARGET_ORDER:
    t = TARGETS[name]
    best = best_for(name)
    if not best:
        lines.append(f'| {name} | {t["track"]} | none | none | 0 | 0 | 0 |')
        continue
    summary['targets'][name] = {'target': t, 'winner': best, 'state': TARGET_STATE.get(name)}
    lines.append(
        f'| {name} | {t["track"]} | {best["tag"]} | {best.get("policy_kind")} | '
        f'{float(best.get("tau") or 0.0):.3f} | {float(best.get("accepted_draft_tokens_per_verify") or 0.0):.2f} | '
        f'{float(best.get("offline_projected_tps") or 0.0):.1f} |'
    )

lines.extend(['', '## Local Mac Gates', ''])
lines.extend([
    'These are deliberately local. Colab cannot prove the Metal runtime accepts drafts or beats the 30 tps baseline.',
    '',
    '1. Build or fetch the target GGUF locally.',
    '2. Build a fresh kernel profile for that GGUF.',
    '3. Run trace-dispatch with the winning head and inspect draft_accepted/draft_rejected.',
    '4. Only after acceptance is real, run clean paired benches.',
    '5. Promote only a stack that beats locked predec baseline under clean bench conditions.',
    '',
])

locked = env_line(QWEN_LOCKED_ENV)
for name in RUN_TARGET_ORDER:
    best = best_for(name)
    if not best:
        continue
    t = TARGETS[name]
    gguf = f'models/{t["gguf_name"]}'
    profile = f'profiles/{t["profile_name"]}'
    head = best['head']
    lines.extend([
        f'### {name} local trace',
        '',
        '~~~bash',
        f'{locked} ./target/release/dismantle generate --trace-dispatch --weights {gguf} --prompt "Once upon a time" --max-new-tokens 64 --temperature 0.0 --kernel-profile {profile} --speculate eagle5 --verify-window 4 --eagle5-head {head}',
        '~~~',
        '',
        f'### {name} paired bench after trace acceptance is nonzero',
        '',
        '~~~bash',
        f'{locked} WEIGHTS={gguf} PROFILE={profile} EAGLE5_HEAD={head} TRIALS=10 TOKENS=128 bash tools/bench/eagle5_paired_bench.sh',
        '~~~',
        '',
    ])

lines.extend(['## Export Contents', ''])
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
heads_out = EXPORT_ROOT / 'heads'
meta_out = EXPORT_ROOT / 'metadata'
heads_out.mkdir(parents=True, exist_ok=True)
meta_out.mkdir(parents=True, exist_ok=True)
export_manifest = {'schema': 'dismantle-maximal-spec-export-v1', 'files': {}, 'missing': {}}

for name in RUN_TARGET_ORDER:
    best = best_for(name)
    if not best:
        continue
    src = Path(best['head'])
    dst = heads_out / f'{name}_{best["tag"]}.safetensors'
    if src.exists():
        shutil.copy2(src, dst)
        export_manifest['files'][f'{name}_winner_head'] = {'source': str(src), 'exported': str(dst), 'bytes': dst.stat().st_size, 'sha256': sha256_file(dst)}
    for key in ['tau_path', 'frontier_path']:
        src = Path(best[key])
        if src.exists():
            dst = meta_out / f'{name}_{best["tag"]}_{src.name}'
            shutil.copy2(src, dst)
            export_manifest['files'][f'{name}_{key}'] = {'source': str(src), 'exported': str(dst), 'bytes': dst.stat().st_size}
    state = TARGET_STATE.get(name, {})
    for qname, qpath in state.get('quant', {}).items():
        src = Path(qpath)
        if src.exists() and src.is_file():
            dst = meta_out / f'{name}_{src.name}'
            shutil.copy2(src, dst)
            export_manifest['files'][f'{name}_{qname}'] = {'source': str(src), 'exported': str(dst), 'bytes': dst.stat().st_size}

summary_path = LAB_ROOT / 'maximal_spec_summary.md'
json_summary_path = LAB_ROOT / 'maximal_spec_summary.json'
summary_path.write_text('\n'.join(lines) + '\n')
write_json_atomic(json_summary_path, summary)
shutil.copy2(summary_path, meta_out / summary_path.name)
shutil.copy2(json_summary_path, meta_out / json_summary_path.name)
shutil.copy2(LAB_ROOT / 'leaderboard.json', meta_out / 'leaderboard.json')

manifest_path = EXPORT_ROOT / 'manifest.json'
write_json_atomic(manifest_path, export_manifest)
zip_path = EXPORT_ROOT.with_suffix('.zip')
if RUN_EXPORT:
    zip_dir(EXPORT_ROOT, zip_path)

record_stage('summary_export', summary_path=str(summary_path), manifest_path=str(manifest_path), zip_path=str(zip_path), size=zip_path.stat().st_size if zip_path.exists() else 0)
print('\n'.join(lines))
print(f'\nsummary: {summary_path}')
print(f'json: {json_summary_path}')
print(f'export: {EXPORT_ROOT}')
print(f'zip: {zip_path if zip_path.exists() else "not written"}')


In [ ]:
# Cell 6B - Safe handoff export mirror (restart-safe; no training, no checkpoint mutation)

# You can run this after a Colab runtime restart. It mounts Drive, refreshes
# /content/dismantle to the notebook branch, then copies already-finished
# artifacts into a separate export folder. It does not rerun training and does
# not mutate any checkpoint directory.

from pathlib import Path
import json, os, shutil, subprocess, time, hashlib, zipfile

REPO_URL_SAFE = globals().get('REPO_URL', 'https://github.com/joshuahickscorp/dismantle.git')
BRANCH_SAFE = globals().get('BRANCH', 'codex/maximal-spec-colab')
REPO_DIR_SAFE = Path(globals().get('REPO_DIR', '/content/dismantle'))

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print(f'[safe-export] Drive mount skipped/failed: {e}')


def _run(cmd):
    print('$', ' '.join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), check=True)

if not (REPO_DIR_SAFE / '.git').exists():
    if REPO_DIR_SAFE.exists():
        shutil.rmtree(REPO_DIR_SAFE)
    _run(['git', 'clone', '--depth', '1', '--branch', BRANCH_SAFE, REPO_URL_SAFE, REPO_DIR_SAFE])
else:
    _run(['git', '-C', REPO_DIR_SAFE, 'fetch', 'origin', BRANCH_SAFE, '--depth', '1'])
    _run(['git', '-C', REPO_DIR_SAFE, 'checkout', BRANCH_SAFE])
    _run(['git', '-C', REPO_DIR_SAFE, 'reset', '--hard', f'origin/{BRANCH_SAFE}'])

os.chdir(REPO_DIR_SAFE)
HEAD_SHA_SAFE = subprocess.check_output(['git', '-C', str(REPO_DIR_SAFE), 'rev-parse', '--short', 'HEAD'], text=True).strip()
print(f'[safe-export] repo={REPO_DIR_SAFE} branch={BRANCH_SAFE} sha={HEAD_SHA_SAFE}')

DRIVE_ROOT_SAFE = globals().get('DRIVE_ROOT', Path('/content/drive/MyDrive/dismantle'))
LAB_ROOT_SAFE = globals().get('LAB_ROOT', DRIVE_ROOT_SAFE / 'maximal_spec_500u')
SAFE_EXPORT_ROOT = DRIVE_ROOT_SAFE / 'dismantle_export' / 'maximal_spec_500u'
SAFE_EXPORT_TOP_N = int(globals().get('SAFE_EXPORT_TOP_N', 30))
SAFE_EXPORT_ZIP = bool(globals().get('SAFE_EXPORT_ZIP', False))


def _load_json(path, default):
    path = Path(path)
    try:
        return json.loads(path.read_text())
    except FileNotFoundError:
        return default


def _write_json_atomic(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(payload, indent=2, sort_keys=True) + '\n')
    os.replace(tmp, path)


def _sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def _copy_file(src, dst, key, manifest, sha=False):
    src = Path(src)
    dst = Path(dst)
    if not src.exists() or not src.is_file():
        manifest['missing'][key] = str(src)
        return None
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_suffix(dst.suffix + '.tmp')
    shutil.copy2(src, tmp)
    os.replace(tmp, dst)
    info = {
        'source': str(src),
        'exported': str(dst),
        'bytes': int(dst.stat().st_size),
    }
    if sha:
        info['sha256'] = _sha256_file(dst)
    manifest['files'][key] = info
    return info


def _zip_dir(src_dir, zip_path):
    src_dir = Path(src_dir)
    zip_path = Path(zip_path)
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for path in sorted(src_dir.rglob('*')):
            if path.is_file() and path != zip_path:
                zf.write(path, path.relative_to(src_dir))

leaderboard_path = LAB_ROOT_SAFE / 'leaderboard.json'
leaderboard = _load_json(leaderboard_path, {'rows': []})
rows = leaderboard.get('rows', []) if isinstance(leaderboard, dict) else []
rows = sorted(
    rows,
    key=lambda r: (float(r.get('offline_projected_tps') or 0.0), float(r.get('tau') or 0.0)),
    reverse=True,
)

manifest = {
    'schema': 'dismantle-maximal-spec-safe-export-v1',
    'created_at_unix': int(time.time()),
    'repo_sha': HEAD_SHA_SAFE,
    'lab_root': str(LAB_ROOT_SAFE),
    'export_root': str(SAFE_EXPORT_ROOT),
    'top_n': SAFE_EXPORT_TOP_N,
    'note': 'Safe mirror only: copies summaries, leaderboard rows, eval JSON, quant metadata, and selected final heads. It does not mutate checkpoints or rerun training.',
    'files': {},
    'missing': {},
    'leaderboard_rows': [],
}

# Core run metadata.
meta_dir = SAFE_EXPORT_ROOT / 'metadata'
_copy_file(leaderboard_path, meta_dir / 'leaderboard.json', 'leaderboard', manifest)
for name in ['maximal_spec_summary.md', 'maximal_spec_summary.json']:
    _copy_file(LAB_ROOT_SAFE / name, meta_dir / name, name, manifest)
for progress_path in sorted(LAB_ROOT_SAFE.glob('progress*.json')):
    _copy_file(progress_path, meta_dir / progress_path.name, f'progress/{progress_path.name}', manifest)
for prior_manifest in [DRIVE_ROOT_SAFE / 'maximal_spec_500u_export' / 'manifest.json', LAB_ROOT_SAFE / 'export_manifest.json']:
    if prior_manifest.exists():
        _copy_file(prior_manifest, meta_dir / prior_manifest.name, f'prior/{prior_manifest.name}', manifest)

# Selected heads and their tau/frontier eval artifacts. By default this catches
# every row shown by the leaderboard, not just the winner, but avoids corpus shards.
selected = rows[:SAFE_EXPORT_TOP_N]
for rank, row in enumerate(selected, start=1):
    target = str(row.get('target') or 'unknown')
    tag = str(row.get('tag') or f'rank_{rank}')
    safe_tag = ''.join(c if c.isalnum() or c in '._-+' else '_' for c in tag)
    manifest['leaderboard_rows'].append({
        'rank': rank,
        'target': target,
        'tag': tag,
        'tau': row.get('tau'),
        'accepted_draft_tokens_per_verify': row.get('accepted_draft_tokens_per_verify'),
        'offline_projected_tps': row.get('offline_projected_tps'),
        'policy_kind': row.get('policy_kind'),
    })
    head = row.get('head')
    if head:
        _copy_file(head, SAFE_EXPORT_ROOT / 'heads' / target / f'{rank:02d}_{safe_tag}.safetensors', f'heads/{target}/{tag}', manifest, sha=True)
    for key in ['tau_path', 'frontier_path']:
        value = row.get(key)
        if value:
            _copy_file(value, SAFE_EXPORT_ROOT / 'eval' / target / safe_tag / Path(value).name, f'eval/{target}/{tag}/{Path(value).name}', manifest)

# Small per-target calibration/quant artifacts from Cell 3. This intentionally
# skips corpora and checkpoints except the selected final heads above.
artifact_root = LAB_ROOT_SAFE / 'artifacts'
for target_dir in sorted(p for p in artifact_root.glob('*') if p.is_dir()):
    for artifact in sorted(target_dir.glob('*')):
        if artifact.is_file() and artifact.suffix.lower() in {'.json', '.npz'}:
            _copy_file(artifact, SAFE_EXPORT_ROOT / 'artifacts' / target_dir.name / artifact.name, f'artifacts/{target_dir.name}/{artifact.name}', manifest)

manifest_path = SAFE_EXPORT_ROOT / 'manifest.json'
_write_json_atomic(manifest_path, manifest)
zip_path = SAFE_EXPORT_ROOT.with_suffix('.zip')
if SAFE_EXPORT_ZIP:
    _zip_dir(SAFE_EXPORT_ROOT, zip_path)
    manifest['zip_path'] = str(zip_path)
    manifest['zip_bytes'] = int(zip_path.stat().st_size)
    _write_json_atomic(manifest_path, manifest)

copied = len(manifest['files'])
missing = len(manifest['missing'])
total_bytes = sum(item['bytes'] for item in manifest['files'].values())
print(f'SAFE_EXPORT_ROOT={SAFE_EXPORT_ROOT}')
print(f'copied={copied} missing={missing} total={total_bytes/1e9:.2f} GB')
print(f'manifest={manifest_path}')
if SAFE_EXPORT_ZIP:
    print(f'zip={zip_path} ({zip_path.stat().st_size/1e9:.2f} GB)')
print('top exported rows:')
for r in manifest['leaderboard_rows'][:10]:
    print(f"  #{r['rank']:02d} {r['target']} {r['tag']} tau={r.get('tau')} accepted={r.get('accepted_draft_tokens_per_verify')} tps={r.get('offline_projected_tps')}")


In [ ]:
# Cell 7 - Tau8 rollout fine-tune from leaderboard winners (new checkpoint dirs only)

# This is the first post-baseline training track from the research notes:
# EAGLE-3/HASS-style training-time-test rollout, but checkpoint-compatible
# with the current Rust loader. It warm-starts from the best baseline head and
# writes new tau8_* checkpoint folders. Existing heads are never modified.

from pathlib import Path
import json, shutil, time, sys

TAU8_RUN_TRAIN = globals().get('TAU8_RUN_TRAIN', True)
TAU8_RUN_EVAL = globals().get('TAU8_RUN_EVAL', True)
TAU8_TARGET_ORDER = globals().get('TAU8_TARGET_ORDER', ['q1p5'])
TAU8_TOP_BASELINES_PER_TARGET = int(globals().get('TAU8_TOP_BASELINES_PER_TARGET', 1))


def _json_load(path, default):
    try:
        return json.loads(Path(path).read_text())
    except FileNotFoundError:
        return default


def _json_write_atomic(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(payload, indent=2, sort_keys=True) + '\n')
    os.replace(tmp, path)


def _read_head_meta(head_path):
    meta = {}
    try:
        from safetensors import safe_open
        with safe_open(str(head_path), framework='pt', device='cpu') as f:
            meta = f.metadata() or {}
    except Exception as e:
        print(f'[tau8] WARN metadata read failed for {head_path}: {e}')
    return meta


def _baseline_rows_for(name, k):
    rows = list(globals().get('LEADERBOARD', []))
    if not rows:
        rows = _json_load(LAB_ROOT / 'leaderboard.json', {}).get('rows', [])
    rows = [r for r in rows if r.get('target') == name and r.get('head')]
    rows.sort(
        key=lambda r: (float(r.get('offline_projected_tps') or 0.0), float(r.get('tau') or 0.0)),
        reverse=True,
    )
    return rows[:k]


def _copy_warm_start(src_ckpt_dir, dst_ckpt_dir):
    src_latest = Path(src_ckpt_dir) / 'latest.npz'
    dst_latest = Path(dst_ckpt_dir) / 'latest.npz'
    if dst_latest.exists():
        return True
    if not src_latest.exists():
        print(f'[tau8] WARN no warm-start latest.npz at {src_latest}; training cold')
        return False
    dst_ckpt_dir.mkdir(parents=True, exist_ok=True)
    tmp = dst_latest.with_suffix(dst_latest.suffix + '.tmp')
    shutil.copy2(src_latest, tmp)
    os.replace(tmp, dst_latest)
    print(f'[tau8] warm-start copied {src_latest} -> {dst_latest}')
    return True

TAU8_SPECS = globals().get('TAU8_SPECS', [
    {
        'name': 'rollout_d5_w015_p075_g090_lr1e-4',
        'epochs': 4,
        'lr': 1e-4,
        'rollout_loss_weight': 0.15,
        'rollout_depth': 5,
        'rollout_starts_per_batch': 4,
        'rollout_draft_prob': 0.75,
        'rollout_depth_gamma': 0.90,
        'calib_loss_weight': 0.12,
        'residual_delta_loss_weight': 0.01,
    },
    {
        'name': 'rollout_d8_w010_p050_g090_lr8e-5',
        'epochs': 4,
        'lr': 8e-5,
        'rollout_loss_weight': 0.10,
        'rollout_depth': 8,
        'rollout_starts_per_batch': 2,
        'rollout_draft_prob': 0.50,
        'rollout_depth_gamma': 0.90,
        'calib_loss_weight': 0.12,
        'residual_delta_loss_weight': 0.015,
    },
    {
        'name': 'rollout_d5_hardneg_w020_p100_lr8e-5',
        'epochs': 3,
        'lr': 8e-5,
        'rollout_loss_weight': 0.20,
        'rollout_depth': 5,
        'rollout_starts_per_batch': 4,
        'rollout_draft_prob': 1.00,
        'rollout_depth_gamma': 0.85,
        'calib_loss_weight': 0.16,
        'residual_delta_loss_weight': 0.02,
    },
])

TAU8_HEADS = {name: [] for name in TAU8_TARGET_ORDER}
if TAU8_RUN_TRAIN:
    for name in TAU8_TARGET_ORDER:
        if name not in TARGET_STATE:
            print(f'[tau8] skip {name}: TARGET_STATE missing; run Cell 2R/3 first')
            continue
        state = TARGET_STATE[name]
        target = state['target']
        baselines = _baseline_rows_for(name, TAU8_TOP_BASELINES_PER_TARGET)
        if not baselines:
            print(f'[tau8] skip {name}: no leaderboard baseline head')
            continue
        for base_rank, base in enumerate(baselines, start=1):
            base_head = Path(base['head'])
            base_tag = base_head.parent.name
            meta = _read_head_meta(base_head)
            nb = int(meta.get('num_blocks', '1'))
            hh = int(meta.get('n_heads', '16'))
            ff = float(meta.get('ff_mult', '4.0'))
            for spec in TAU8_SPECS:
                tag = f'{name}_tau8_{spec["name"]}_from_{base_tag}'
                ckpt_dir = Path(state['artifact_dir']) / 'checkpoints' / tag
                head = ckpt_dir / 'head_final.safetensors'
                if head.exists():
                    print(f'[tau8:{name}] skip existing {head}')
                    TAU8_HEADS[name].append(head)
                    continue
                _copy_warm_start(base_head.parent, ckpt_dir)
                batch = min(int(target.get('rollout_batch', 56 if BIG_GPU else 24)), 56 if BIG_GPU else 24)
                cmd = [
                    sys.executable, '-u', 'colab/eagle5_train_pytorch.py',
                    '--corpus-dir', state['corpus_dir'],
                    '--frozen', state['local_frozen'],
                    '--ckpt-dir', str(ckpt_dir),
                    '--epochs', str(spec['epochs']),
                    '--batch-size', str(batch),
                    '--seq-len', str(target['train_seq_len'] if 'train_seq_len' in target else 16),
                    '--lr', str(spec['lr']),
                    '--num-blocks', str(nb),
                    '--head-heads', str(hh),
                    '--head-ff-mult', str(ff),
                    '--capture-layer', str(target['capture_layer']),
                    '--max-rows', str(target['train_max_rows']),
                    '--max-row-tokens', str(target['train_max_row_tokens']),
                    '--sparsity-head', 'off',
                    '--seed', str(1000 + base_rank),
                    '--calib-loss-weight', str(spec['calib_loss_weight']),
                    '--residual-delta-loss-weight', str(spec['residual_delta_loss_weight']),
                    '--rollout-loss-weight', str(spec['rollout_loss_weight']),
                    '--rollout-depth', str(spec['rollout_depth']),
                    '--rollout-starts-per-batch', str(spec['rollout_starts_per_batch']),
                    '--rollout-draft-prob', str(spec['rollout_draft_prob']),
                    '--rollout-depth-gamma', str(spec['rollout_depth_gamma']),
                    '--save-safetensors',
                ]
                print(f'\n=== [tau8:{name}] train {tag} batch={batch} warm={base_tag}')
                run_with_heartbeat(cmd, label=f'tau8_{name}_{spec["name"]}', interval_sec=60)
                if not head.exists():
                    raise FileNotFoundError(f'tau8 head missing after train: {head}')
                TAU8_HEADS[name].append(head)
                record_stage(
                    f'tau8_train_{tag}',
                    path=str(head),
                    size=head.stat().st_size,
                    source=str(base_head),
                    spec=spec,
                )
else:
    print('TAU8_RUN_TRAIN=False; collecting existing tau8 heads only.')
    for name in TAU8_TARGET_ORDER:
        ckpt_root = Path(TARGET_STATE[name]['artifact_dir']) / 'checkpoints'
        TAU8_HEADS[name] = sorted(p for p in ckpt_root.glob(f'{name}_tau8_*/head_final.safetensors'))

TAU8_LEADERBOARD = []
if TAU8_RUN_EVAL:
    if 'eval_head' not in globals():
        print('[tau8] eval_head is not defined; run Cell 5 first, then rerun this cell with TAU8_RUN_TRAIN=False')
    else:
        for name, heads in TAU8_HEADS.items():
            for head in heads:
                TAU8_LEADERBOARD.append(eval_head(name, head))
else:
    print('TAU8_RUN_EVAL=False; skipping tau/frontier eval.')

if TAU8_LEADERBOARD:
    existing_rows = _json_load(LAB_ROOT / 'leaderboard.json', {}).get('rows', [])
    by_key = {(r.get('target'), r.get('tag')): r for r in existing_rows}
    for row in TAU8_LEADERBOARD:
        by_key[(row.get('target'), row.get('tag'))] = row
    merged = sorted(
        by_key.values(),
        key=lambda r: (float(r.get('offline_projected_tps') or 0.0), float(r.get('tau') or 0.0)),
        reverse=True,
    )
    globals()['LEADERBOARD'] = merged
    leaderboard_path = LAB_ROOT / 'leaderboard.json'
    _json_write_atomic(leaderboard_path, {'schema': 'dismantle-maximal-spec-leaderboard-v1', 'rows': merged})
    record_stage('tau8_leaderboard', path=str(leaderboard_path), rows=len(merged))
    print('\n=== Tau8 leaderboard rows ===')
    for r in TAU8_LEADERBOARD:
        print(f"{r['target']:7s} {r['tag'][:64]:64s} tau={r.get('tau')} acc1={r.get('depth1_accept_rate')} accepted={r.get('accepted_draft_tokens_per_verify')} offline_tps={r.get('offline_projected_tps')} policy={r.get('policy_kind')}")
